In [ ]:
!uv add -r rag-requirements.txt

In [2]:
from langsmith import Client
client = Client()

cod_prompt = client.pull_prompt(
    "teddynote/chain-of-density-prompt",
    dangerously_pull_public_prompt=True
)

cod_prompt.pretty_print()

================================ System Message ================================

As an expert copy-writer, you will write increasingly concise, entity-dense summaries of the user provided {content_category}. The initial summary should be under {max_words} words and contain {entity_range} informative Descriptive Entities from the {content_category}.

A Descriptive Entity is:
- Relevant: to the main story.
- Specific: descriptive yet concise (5 words or fewer).
- Faithful: present in the {content_category}.
- Anywhere: located anywhere in the {content_category}.

# Your Summarization Process
- Read through the {content_category} and the all the below sections to get an understanding of the task.
- Pick {entity_range} informative Descriptive Entities from the {content_category} (";" delimited, do not add spaces).
- In your output JSON list of dictionaries, write an initial summary of max {max_words} words containing the Entities.
- You now have `[{"missing_entities": "...", "denser_summa

In [5]:
import textwrap
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import SimpleJsonOutputParser
from langchain_community.document_loaders import PyPDFLoader

client = Client()

loader = PyPDFLoader("data/Truth.pdf")
docs = loader.load()
docs = docs[0:9]

cod_chain_inputs ={
    "content": lambda d: d.get("content"),
    "content_category": lambda d: d.get("content_category", "Article"),
    "entity_range": lambda d: d.get("entity_range", "1-3"),
    "max_words": lambda d: int(d.get("max_words", 80)),
    "iterations": lambda d: int(d.get("iterations", 5)),
}

cod_prompt = client.pull_prompt("teddynote/chain-of-density-prompt",
                                dangerously_pull_public_prompt=True)

cod_chain = (
    cod_chain_inputs
    | cod_prompt
    | ChatOpenAI(temperature=0.1, model="gpt-4o-mini")
    | SimpleJsonOutputParser()
)

cod_final_summary_chain = cod_chain |(
    lambda output: output[-1].get(
        "denser_summary", '오류: 마지막 딕셔너리에 "denser_summary" 키가 없습니다.'
    )
)

content = docs[1].page_content
print(content)

매일일보 조정훈 인턴기자 기사 - RAG/Retriever 실습용 요약
2
수록 기사
 ID
기사 제목
A01
국감 앞두고 공세 수위 올리는 野…장외 집회는 '신중론'
A02
[미리보는 국감 2026] 운영위·재경위·행안위 최대 쟁점…인사·부동산·경찰개혁
A03
미중 정상회담서 웃었지만…AI·대만·이란 놓고는 여전한 입장차
A04
李대통령, 순방 귀국길 알래스카서 '즉석 차담회'
A05
정점식 "일도 못하는 무능한 대통령…국민 한숨 이어져"
A06
나경원 "DMZ 지뢰 폭발사고 진상 조사단, 야당 추천 전문가 포함해야"
A07
국힘, 영화 '암살자(들)'에 "명백한 역사 왜곡"
실습 팁
기사마다 제목, 요약, 키워드, 테스트 질문을 같은 구조로 배치했습니다. PDFLoader로 불러온 뒤 페이지 단위 또는
RecursiveCharacterTextSplitter로 청크를 나누고, metadata에 article_id와 source를 넣어 검색해보면
좋습니다.


In [8]:
results: list[dict[str, str]] = []
for partial_json in cod_chain.stream(
    {"content": content, "content_category": "Article"}
):
    results = partial_json

    print(results, end="\r", flush=True)

total_summaries = len(results)
print("\n")

i = 1
for cod in results:
    added_entities = ",".join(
        [
            ent.strip()
            for ent in cod.get(
                "missing_entities", 'ERR: "missing_entities" key no found'
            ).split(";")
        ]
    )
    summary = cod.get("denser_summary", 'ERR: missing key "denser_summary"')
    print(
        f"### CoD Summary {i}/{total_summaries}, 추가된 엔티티(entity): {added_entities}"
        + "\n"
    )

    print(textwrap.fill(summary, width=80)+ "\n")
    i += 1

print("\n======== [최종 요약] ========\n")
print(summary)


[{'missing_entities': '국감;AI;대만', 'denser_summary': '이 기사에서는 국감과 관련된 여러 주제를 다루고 있습니다. 특히, 국감 앞두고 공세 수위를 올리는 야당의 움직임과 AI, 대만, 이란에 대한 미중 정상회담의 입장차를 강조합니다. 또한, 장외 집회에 대한 신중론과 주요 쟁점으로 인사, 부동산, 경찰개혁을 언급합니다.'}, {'missing_entities': '정점식;나경원', 'denser_summary': '이 기사에서는 국감과 관련된 여러 주제를 다루고 있습니다. 특히, 국감 앞두고 공세 수위를 올리는 야당의 움직임과 AI, 대만, 이란에 대한 미중 정상회담의 입장차를 강조합니다. 정점식과 나경원의 발언도 포함되어 있으며, 장외 집회에 대한 신중론과 주요 쟁점으로 인사, 부동산, 경찰개혁을 언급합니다.'}, {'missing_entities': "국힘;영화 '암살자(들)'", 'denser_summary': "이 기사에서는 국감과 관련된 여러 주제를 다루고 있습니다. 특히, 국감 앞두고 공세 수위를 올리는 야당의 움직임과 AI, 대만, 이란에 대한 미중 정상회담의 입장차를 강조합니다. 정점식과 나경원의 발언, 국힘의 영화 '암살자(들)'에 대한 역사 왜곡 주장도 포함되어 있으며, 장외 집회에 대한 신중론과 주요 쟁점으로 인사, 부동산, 경찰개혁을 언급합니다."}, {'missing_entities': '즉석 차담회;무능한 대통령', 'denser_summary': "이 기사에서는 국감과 관련된 여러 주제를 다루고 있습니다. 특히, 국감 앞두고 공세 수위를 올리는 야당의 움직임과 AI, 대만, 이란에 대한 미중 정상회담의 입장차를 강조합니다. 정점식과 나경원의 발언, 국힘의 영화 '암살자(들)'에 대한 역사 왜곡 주장, 이 대통령의 알래스카 즉석 차담회와 무능한 대통령에 대한 비판도 포함되어 있으며, 장외 집회에 대한 신중론과 주요 쟁점으로 인사, 부동산, 경찰개혁을 언급합니다."}, {'missing_entiti